In [3]:
# Import libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3


df = pd.read_csv('C:/Users/Fatima zahra/OneDrive/Desktop/MyProjects/causes-of-death-worldwide/annual-number-of-deaths-by-cause.csv')
conn = sqlite3.connect('causes_of_death.db')
df.to_sql('causes_of_death', conn, if_exists='replace', index=False)


test = pd.read_sql_query("SELECT COUNT(*) as total FROM causes_of_death", conn)
print(f"📊 Total rows in database: {test['total'][0]}")


print(df.shape)
df.head()

📊 Total rows in database: 8254
(8254, 36)


,Entity,Code,Year,Number of executions (Amnesty International),Deaths - Meningitis - Sex: Both - Age: All Ages (Number),Deaths - Neoplasms - Sex: Both - Age: All Ages (Number),"Deaths - Fire, heat, and hot substances - Sex: Both - Age: All Ages (Number)",Deaths - Malaria - Sex: Both - Age: All Ages (Number),Deaths - Drowning - Sex: Both - Age: All Ages (Number),Deaths - Interpersonal violence - Sex: Both - Age: All Ages (Number),...,Deaths - Protein-energy malnutrition - Sex: Both - Age: All Ages (Number),Terrorism (deaths),Deaths - Cardiovascular diseases - Sex: Both - Age: All Ages (Number),Deaths - Chronic kidney disease - Sex: Both - Age: All Ages (Number),Deaths - Chronic respiratory diseases - Sex: Both - Age: All Ages (Number),Deaths - Cirrhosis and other chronic liver diseases - Sex: Both - Age: All Ages (Number),Deaths - Digestive diseases - Sex: Both - Age: All Ages (Number),Deaths - Acute hepatitis - Sex: Both - Age: All Ages (Number),Deaths - Alzheimer's disease and other dementias - Sex: Both - Age: All Ages (Number),Deaths - Parkinson's disease - Sex: Both - Age: All Ages (Number)
0,Afghanistan,AFG,2007,15,2933.0,15925.0,481.0,393.0,2127.0,3657.0,...,2439.0,1199.0,53962.0,4490.0,7222.0,3346.0,6458.0,3437.0,1402.0,450.0
1,Afghanistan,AFG,2008,17,2731.0,16148.0,462.0,255.0,1973.0,3785.0,...,2231.0,1092.0,54051.0,4534.0,7143.0,3316.0,6408.0,3005.0,1424.0,455.0
2,Afghanistan,AFG,2009,0,2460.0,16383.0,448.0,239.0,1852.0,3874.0,...,1998.0,1065.0,53964.0,4597.0,7045.0,3291.0,6358.0,2663.0,1449.0,460.0
3,Afghanistan,AFG,2011,2,2327.0,17094.0,448.0,390.0,1775.0,4170.0,...,1805.0,1525.0,54347.0,4785.0,6916.0,3318.0,6370.0,2365.0,1508.0,473.0
4,Afghanistan,AFG,2012,14,2254.0,17522.0,445.0,94.0,1716.0,4245.0,...,1667.0,3521.0,54868.0,4846.0,6878.0,3353.0,6398.0,2264.0,1544.0,482.0


In [8]:
# Clean column names
df.columns = df.columns.str.replace('Deaths - ', '', regex=False)
df.columns = df.columns.str.replace(' - Sex: Both - Age: All Ages (Number)', '', regex=False)
df.columns = df.columns.str.strip()

# Reload to SQLite with clean names
df.to_sql('causes_of_death', conn, if_exists='replace', index=False)

# Verify
print(df.columns.tolist())
print(pd.read_sql("SELECT * FROM causes_of_death LIMIT 2", conn))

['Entity', 'Code', 'Year', 'Meningitis', 'Neoplasms', 'Fire, heat, and hot substances', 'Malaria', 'Drowning', 'Interpersonal violence', 'HIV/AIDS', 'Drug use disorders', 'Tuberculosis', 'Road injuries', 'Maternal disorders', 'Lower respiratory infections', 'Neonatal disorders', 'Alcohol use disorders', 'Exposure to forces of nature', 'Diarrheal diseases', 'Environmental heat and cold exposure', 'Nutritional deficiencies', 'Self-harm', 'Conflict and terrorism', 'Diabetes mellitus', 'Poisonings', 'Protein-energy malnutrition', 'Cardiovascular diseases', 'Chronic kidney disease', 'Chronic respiratory diseases', 'Cirrhosis and other chronic liver diseases', 'Digestive diseases', 'Acute hepatitis', "Alzheimer's disease and other dementias", "Parkinson's disease"]
        Entity Code  Year  Meningitis  Neoplasms  \
0  Afghanistan  AFG  2007      2933.0    15925.0   
1  Afghanistan  AFG  2008      2731.0    16148.0   

   Fire, heat, and hot substances  Malaria  Drowning  Interpersonal viole

In [6]:
df.isnull().sum()

Entity                                             0
Code                                            2048
Year                                               0
Number of executions (Amnesty International)    7987
Meningitis                                       244
Neoplasms                                        244
Fire, heat, and hot substances                   244
Malaria                                          244
Drowning                                         244
Interpersonal violence                           244
HIV/AIDS                                         244
Drug use disorders                               244
Tuberculosis                                     244
Road injuries                                    244
Maternal disorders                               244
Lower respiratory infections                     244
Neonatal disorders                               244
Alcohol use disorders                            244
Exposure to forces of nature                  

In [7]:
# Drop columns that are mostly empty and not useful
df = df.drop(columns=['Number of executions (Amnesty International)', 'Terrorism (deaths)'])

# Fill missing death counts with 0
death_columns = df.columns[3:]
df[death_columns] = df[death_columns].fillna(0)

# Verify no more missing values
print("Remaining missing values:")
print(df.isnull().sum())
print(f"\n✅ Dataset shape after cleaning: {df.shape}")

Remaining missing values:
Entity                                           0
Code                                          2048
Year                                             0
Meningitis                                       0
Neoplasms                                        0
Fire, heat, and hot substances                   0
Malaria                                          0
Drowning                                         0
Interpersonal violence                           0
HIV/AIDS                                         0
Drug use disorders                               0
Tuberculosis                                     0
Road injuries                                    0
Maternal disorders                               0
Lower respiratory infections                     0
Neonatal disorders                               0
Alcohol use disorders                            0
Exposure to forces of nature                     0
Diarrheal diseases                               0
Envir

Find all countries (Entity) where the average Cardiovascular diseases deaths across all years is above 100,000. Show Entity and average, ordered highest first.

In [14]:
query  = """ 

SELECT Entity , ROUND(AVG("Cardiovascular diseases"), 0) AS avg_cardio
FROM causes_of_death
GROUP BY Entity
HAVING avg_cardio > 100000
ORDER BY avg_cardio DESC

"""


pd.read_sql(query, conn)

,Entity,avg_cardio
0,World,14932656.0
1,G20,10864386.0
2,Asia,7856399.0
3,World Bank Upper Middle Income,6396381.0
4,East Asia & Pacific - World Bank region,4951922.0
...,...,...
74,Spain,124229.0
75,Philippines,124157.0
76,Myanmar,117414.0
77,Mexico,108150.0


Find countries that appear in MORE than 20 years of data AND have average Malaria deaths above 10,000. Show Entity, year count, and average Malaria deaths.

In [18]:
query = """

SELECT Entity , COUNT(*) AS years , ROUND(AVG(Malaria),0) AS avg_malaria
FROM causes_of_death
GROUP BY Entity 
HAVING COUNT(*) > 20 AND AVG(Malaria) > 10000

"""
pd.read_sql(query, conn)

,Entity,years,avg_malaria
0,Africa,30,722874.0
1,African Region,30,715423.0
2,African Union,30,722874.0
3,Angola,30,10569.0
4,Asia,30,118038.0
5,Bangladesh,30,11646.0
6,Benin,30,10561.0
7,Burkina Faso,30,31692.0
8,Burundi,30,10692.0
9,Cameroon,30,20470.0


Filter for Year 2019 only (WHERE), then find countries where BOTH HIV/AIDS deaths AND Tuberculosis deaths average above 5,000. Use HAVING for both conditions.

In [31]:
query = """
SELECT Entity , Year , ROUND(AVG("HIV/AIDS"),0) AS avg_HIV_AIDS , ROUND(AVG(Tuberculosis),0) AS avg_tuberculosis
FROM causes_of_death
WHERE Year = 2019 AND Code NOT LIKE 'OWID%'
GROUP BY Entity
HAVING AVG("HIV/AIDS") > 5000
   AND AVG("Tuberculosis") > 5000
   
ORDER BY avg_HIV_AIDS DESC
"""

pd.read_sql(query , conn)

,Entity,Year,avg_HIV_AIDS,avg_tuberculosis
0,South Africa,2019,143851.0,19785.0
1,Nigeria,2019,82270.0,45278.0
2,Mozambique,2019,66304.0,20700.0
3,Kenya,2019,51135.0,16781.0
4,India,2019,46298.0,422634.0
5,China,2019,31746.0,36566.0
6,Tanzania,2019,27125.0,20170.0
7,Ethiopia,2019,26591.0,29874.0
8,Cameroon,2019,23172.0,6551.0
9,Zambia,2019,22540.0,7056.0


Using a CTE for average Cardiovascular deaths per country, JOIN it back to the main table to show each year's data alongside the country's all-time average. Show Entity, Year, that year's deaths, and the country average. Limit to Morocco.

In [40]:
query = """
WITH cte AS (
SELECT Entity ,ROUND(AVG("Cardiovascular diseases"),0) AS avg_cardio
FROM causes_of_death
GROUP BY Entity

)

SELECT c.Entity,
    c.Year,
    c."Cardiovascular diseases" AS this_year_deaths,
    ca.avg_cardio AS country_avg

FROM causes_of_death AS c
JOIN cte AS ca ON c.Entity = ca.Entity
WHERE c.Entity = 'Morocco'
ORDER BY c.Year DESC
"""

pd.read_sql(query , conn)

,Entity,Year,this_year_deaths,country_avg
0,Morocco,2019,117034.0,85037.0
1,Morocco,2018,115052.0,85037.0
2,Morocco,2017,112904.0,85037.0
3,Morocco,2016,110845.0,85037.0
4,Morocco,2015,109198.0,85037.0
5,Morocco,2014,108079.0,85037.0
6,Morocco,2013,107171.0,85037.0
7,Morocco,2012,96921.0,85037.0
8,Morocco,2011,94564.0,85037.0
9,Morocco,2010,93006.0,85037.0


Using TWO CTEs: one for average Cardiovascular deaths per country, one for average Diabetes deaths per country. JOIN them to show both side by side. Which country has high cardiovascular AND high diabetes deaths?

In [48]:
query = """

WITH cardio_cte AS (

SELECT Entity , ROUND(AVG("Cardiovascular diseases"),0) AS avg_cardio
FROM causes_of_death
WHERE Code NOT LIKE 'OWID%'
GROUP BY Entity

),
diabetes_cte AS (
 SELECT Entity , ROUND(AVG("Diabetes mellitus"),0) AS avg_diabetes
 FROM causes_of_death
 WHERE Code NOT LIKE 'OWID%'
 GROUP BY Entity
)
SELECT c.Entity, c.avg_cardio AS CARDIO, ca.avg_diabetes AS DIABETES

FROM cardio_cte AS c
JOIN diabetes_cte AS ca 

ON c.Entity = ca.Entity
ORDER BY c.avg_cardio DESC
LIMIT 15
"""
pd.read_sql(query, conn)

,Entity,CARDIO,DIABETES
0,China,3350199.0,115618.0
1,India,1766490.0,159372.0
2,Russia,1130126.0,12322.0
3,United States,881278.0,67688.0
4,Indonesia,452900.0,62952.0
5,Ukraine,435102.0,3191.0
6,Germany,360659.0,20719.0
7,Brazil,319634.0,43859.0
8,Japan,307015.0,8372.0
9,Pakistan,258173.0,28178.0


Create a CTE with total deaths from Conflict and terrorism per country. LEFT JOIN it with a CTE of total Cardiovascular deaths. Show all countries — even those with zero conflict deaths. Order by conflict deaths descending.

In [57]:
query = """ 

WITH conflict AS (
SELECT Entity , ROUND(SUM("Conflict and terrorism"), 0) AS total_conflict
FROM causes_of_death 
WHERE Code NOT LIKE 'OWID%'
GROUP BY Entity
),

cardio AS (
SELECT Entity , ROUND(SUM("Cardiovascular diseases"), 0) AS total_cardio
FROM causes_of_death
WHERE Code NOT LIKE 'OWID%'
GROUP BY Entity
)
SELECT c.Entity ,c.total_conflict , ca.total_cardio
FROM cardio AS ca
LEFT JOIN conflict AS c
ON c.Entity = ca.Entity
ORDER BY c.total_conflict DESC
LIMIT 10

"""

pd.read_sql(query, conn)

,Entity,total_conflict,total_cardio
0,Rwanda,528779.0,278336.0
1,Syria,365293.0,1056981.0
2,Iraq,336979.0,1655522.0
3,Afghanistan,280520.0,1607042.0
4,Burundi,163935.0,262614.0
5,Ethiopia,158564.0,1936154.0
6,Democratic Republic of Congo,116313.0,1989495.0
7,Yemen,95610.0,1110837.0
8,Sudan,92787.0,1887465.0
9,Sri Lanka,82347.0,1149526.0


For year 2019, find all countries where Malaria deaths were above the world average Malaria deaths for that same year. Show Entity and Malaria deaths.

In [59]:
query = """ 
SELECT Entity, Year ,Malaria AS malaria_deaths
FROM causes_of_death
WHERE Code NOT LIKE 'OWID%'
AND Year = 2019
AND Malaria > (

      SELECT AVG("Malaria")
      FROM causes_of_death
      WHERE Year = 2019
)
ORDER BY malaria_deaths DESC
"""
pd.read_sql(query ,conn)

,Entity,Year,malaria_deaths
0,Nigeria,2019,191106.0
1,Democratic Republic of Congo,2019,57160.0
2,India,2019,33372.0
3,Cote d'Ivoire,2019,29398.0
4,Burkina Faso,2019,26305.0


Find the year where global Cardiovascular deaths were the highest. Use a subquery — without just using ORDER BY + LIMIT 1.

In [69]:
query = """
SELECT Year, SUM("Cardiovascular diseases") AS MAX_SUM
FROM causes_of_death 
GROUP BY Year
HAVING SUM("Cardiovascular diseases") =  (
    SELECT MAX(total)
    FROM (
        SELECT SUM("Cardiovascular diseases") AS total
        FROM causes_of_death 
        GROUP BY Year
    )
)

"""
pd.read_sql(query , conn)

,Year,MAX_SUM
0,2019,186591929.0


BOSS: Find countries where Cardiovascular deaths are above the global average AND Malaria deaths are BELOW the global average. These are likely developed countries — high chronic disease, low infectious disease. Use subqueries for both conditions. Filter for Year 2019.

In [74]:
query= """

SELECT Entity, ROUND(SUM("Cardiovascular diseases")) as total_cardio , ROUND(SUM("Malaria")) as total_malaria 
FROM causes_of_death
WHERE Year = 2019
  AND Code NOT LIKE 'OWID%'
GROUP BY Entity 
HAVING SUM("Cardiovascular diseases") > (
    SELECT AVG(Smm)
    FROM (
        SELECT ROUND(SUM("Cardiovascular diseases")) as Smm
        FROM causes_of_death
        WHERE Year = 2019
  AND Code NOT LIKE 'OWID%'
        GROUP BY Entity 
    )
) AND SUM("Malaria") < (
       SELECT AVG(Smm1)
    FROM (
        SELECT ROUND(SUM("Malaria")) as Smm1
        FROM causes_of_death
        WHERE Year = 2019
  AND Code NOT LIKE 'OWID%'
        GROUP BY Entity 
    )
)
ORDER BY total_cardio DESC
LIMIT 10
"""
pd.read_sql(query , conn)

,Entity,total_cardio,total_malaria
0,China,4584273.0,0.0
1,Russia,1004931.0,0.0
2,United States,957455.0,0.0
3,Indonesia,651481.0,637.0
4,Ukraine,449376.0,0.0
5,Brazil,397993.0,118.0
6,Japan,372483.0,0.0
7,Germany,364285.0,0.0
8,Bangladesh,324764.0,110.0
9,Egypt,263873.0,0.0


Show all data for Morocco across all years. Display Entity, Year, Cardiovascular diseases, Malaria, and HIV/AIDS deaths. Order by Year ascending.

In [78]:
query = """ 

SELECT Entity , Year , "Cardiovascular diseases" , Malaria , "HIV/AIDS"
FROM causes_of_death
WHERE Entity = "Morocco"
ORDER BY Year ASC 

"""
pd.read_sql(query , conn)

,Entity,Year,Cardiovascular diseases,Malaria,HIV/AIDS
0,Morocco,1990,57888.0,0.0,103.0
1,Morocco,1991,59719.0,0.0,127.0
2,Morocco,1992,62875.0,0.0,155.0
3,Morocco,1993,65445.0,0.0,187.0
4,Morocco,1994,66526.0,0.0,223.0
5,Morocco,1995,68162.0,0.0,264.0
6,Morocco,1996,69194.0,0.0,308.0
7,Morocco,1997,70916.0,0.0,355.0
8,Morocco,1998,72395.0,0.0,407.0
9,Morocco,1999,74713.0,0.0,466.0


Find all rows where Conflict and terrorism deaths were above 10,000 AND the Year is between 2010 and 2020. Show Entity, Year, and Conflict deaths. Order by deaths descending.

In [80]:
query = """

SELECT Entity , Year, "Conflict and terrorism"
FROM causes_of_death
WHERE Year BETWEEN 2010 AND 2020 
  AND Code NOT LIKE 'OWID%'
AND "Conflict and terrorism" > 10000
ORDER BY "Conflict and terrorism" DESC

"""
pd.read_sql(query , conn)

,Entity,Year,Conflict and terrorism
0,Syria,2014,75356.0
1,Syria,2015,60942.0
2,Syria,2012,60179.0
3,Syria,2016,54433.0
4,Syria,2013,49731.0
5,Iraq,2014,37209.0
6,Syria,2017,31830.0
7,Iraq,2016,31752.0
8,Yemen,2018,30779.0
9,Iraq,2015,29814.0


For each country (Entity), calculate total Cardiovascular diseases deaths across ALL years. Show only countries with more than 5 million total deaths. Order by total descending.

In [88]:
query = """ 

SELECT Entity , ROUND(SUM("Cardiovascular diseases"),0) AS total_cardio
FROM causes_of_death 
WHERE Code NOT LIKE 'OWID%'
GROUP BY Entity 
HAVING SUM("Cardiovascular diseases") > 5000000
ORDER BY total_cardio DESC

"""
pd.read_sql(query , conn)

,Entity,total_cardio
0,China,100505973.0
1,India,52994710.0
2,Russia,33903781.0
3,United States,26438346.0
4,Indonesia,13587012.0
5,Ukraine,13053052.0
6,Germany,10819770.0
7,Brazil,9589019.0
8,Japan,9210437.0
9,Pakistan,7745192.0


For each Year, find the total global Malaria deaths. Only show years where global Malaria deaths exceeded 500,000. Is Malaria increasing or decreasing over time?

In [94]:
query = """

SELECT Year, ROUND(SUM("Malaria"),0) AS Total_Malaria_Deaths
FROM causes_of_death 
WHERE Code NOT LIKE 'OWID%'
GROUP BY Year
HAVING SUM("Malaria") > 500000
ORDER BY Year ASC
"""
pd.read_sql(query, conn)

,Year,Total_Malaria_Deaths
0,1990,840298.0
1,1991,858984.0
2,1992,856415.0
3,1993,862217.0
4,1994,855671.0
5,1995,862625.0
6,1996,872476.0
7,1997,892946.0
8,1998,901338.0
9,1999,893788.0


Using a CTE, first calculate total deaths from ALL of these causes per country: Malaria + HIV/AIDS + Tuberculosis + Neonatal disorders. Call this 'infectious_total'. Then show the top 10 countries with highest infectious disease burden.

In [98]:
query = """

WITH infectious AS (

SELECT Entity , ROUND(SUM(Malaria + "HIV/AIDS" + Tuberculosis + "Neonatal disorders"),0) AS infectious_total
FROM causes_of_death 
WHERE Code NOT LIKE 'OWID%'
GROUP BY Entity

) 
SELECT Entity, infectious_total
FROM infectious
ORDER BY infectious_total DESC
LIMIT 10


"""
pd.read_sql(query, conn)

,Entity,infectious_total
0,India,41626110.0
1,Nigeria,15670400.0
2,Pakistan,10313608.0
3,China,7509256.0
4,Ethiopia,6833268.0
5,Democratic Republic of Congo,6595553.0
6,South Africa,6345061.0
7,Indonesia,5584784.0
8,Tanzania,5026306.0
9,Uganda,4808178.0


OR DO THIS :

In [99]:
query = """

SELECT Entity , infectious_total 
FROM (

SELECT Entity , ROUND(SUM(Malaria + "HIV/AIDS" + Tuberculosis + "Neonatal disorders"),0) AS infectious_total
FROM causes_of_death 
WHERE Code NOT LIKE 'OWID%'
GROUP BY Entity

) 
ORDER BY infectious_total DESC
LIMIT 10


"""
pd.read_sql(query, conn)

,Entity,infectious_total
0,India,41626110.0
1,Nigeria,15670400.0
2,Pakistan,10313608.0
3,China,7509256.0
4,Ethiopia,6833268.0
5,Democratic Republic of Congo,6595553.0
6,South Africa,6345061.0
7,Indonesia,5584784.0
8,Tanzania,5026306.0
9,Uganda,4808178.0


Using TWO CTEs: one for total infectious deaths (Malaria + HIV/AIDS + Tuberculosis) per country, one for total chronic deaths (Cardiovascular diseases + Neoplasms + Diabetes mellitus) per country. JOIN them and find countries where chronic deaths are more than 10x the infectious deaths.

In [100]:
query = """ 

WITH total_infectious AS (

SELECT Entity , ROUND(SUM("Malaria" + "HIV/AIDS" + "Tuberculosis"),0) as infectious
FROM causes_of_death 
WHERE Code NOT LIKE 'OWID%'
GROUP BY Entity

),

total_chronic AS (

SELECT Entity , ROUND(SUM("Cardiovascular diseases" + "Neoplasms" + "Diabetes mellitus"),0) as chronic
FROM causes_of_death 
WHERE Code NOT LIKE 'OWID%'
GROUP BY Entity

)
SELECT c1.Entity , c1.infectious , c2.chronic
FROM total_infectious AS c1
JOIN total_chronic AS c2
ON c1.Entity = c2.Entity
WHERE c2.chronic > c1.infectious * 10


ORDER BY c2.chronic DESC
LIMIT 15
"""
pd.read_sql(query, conn)

,Entity,infectious,chronic
0,China,3155589.0,165035055.0
1,United States,566544.0,47374293.0
2,Russia,880532.0,43352743.0
3,Japan,140897.0,19992050.0
4,Germany,52773.0,18766994.0
5,Brazil,710962.0,16511552.0
6,Ukraine,323656.0,16339112.0
7,Italy,65264.0,12356154.0
8,United Kingdom,26408.0,11980383.0
9,France,99763.0,10120656.0


For year 2019, find countries where Diabetes mellitus deaths are above the global average for that year. Show Entity, diabetes deaths, and the difference from the global average.

In [111]:
query = """ 

SELECT Entity, 
       "Diabetes mellitus" AS diabetes_death, 
        ROUND("Diabetes mellitus" - (
            SELECT AVG("Diabetes mellitus")
            FROM causes_of_death
            WHERE Year = 2019
       ), 0) AS above_global_avg
    
FROM causes_of_death
WHERE Code NOT LIKE 'OWID%' AND Year = 2019
  AND "Diabetes mellitus" > (
      SELECT AVG("Diabetes mellitus")
      FROM causes_of_death
      WHERE Year = 2019
      AND Code NOT LIKE 'OWID%' 
  )
ORDER BY diabetes_death DESC
LIMIT 10

"""
pd.read_sql(query , conn)

,Entity,diabetes_death,above_global_avg
0,India,273089.0,215374.0
1,China,172892.0,115177.0
2,Indonesia,106333.0,48618.0
3,United States,77719.0,20004.0
4,Mexico,73838.0,16123.0
5,Brazil,65366.0,7651.0
6,Pakistan,47693.0,-10022.0
7,Bangladesh,33095.0,-24620.0
8,Vietnam,29391.0,-28324.0
9,Philippines,27680.0,-30035.0


Find the year with the highest global Self-harm deaths. Then show ALL countries' Self-harm deaths for that specific year, ordered highest first.

In [126]:
query = """

SELECT Entity, Year, "Self-harm"
FROM causes_of_death
WHERE Code NOT LIKE 'OWID%'
AND Year = (
    SELECT Year
    FROM causes_of_death
    GROUP BY Year
    ORDER BY SUM("Self-harm") DESC
    LIMIT 1
)
ORDER BY "Self-harm" DESC 
LIMIT 11

"""
pd.read_sql(query, conn)

,Entity,Year,Self-harm
0,China,1999,198900.0
1,India,1999,195957.0
2,Russia,1999,71781.0
3,United States,1999,34091.0
4,Japan,1999,31283.0
5,Ukraine,1999,17862.0
6,France,1999,13639.0
7,Germany,1999,13149.0
8,Pakistan,1999,11210.0
9,Thailand,1999,10680.0


For each row, show Entity, Year, Cardiovascular deaths, AND the average Cardiovascular deaths for that country across all years — side by side. Limit to Morocco and show all years

In [134]:
query = """

SELECT Entity, Year,"Cardiovascular diseases", ROUND(AVG("Cardiovascular diseases") OVER (PARTITION BY Entity), 0) AS avg_cardio
FROM causes_of_death 
WHERE Entity = 'Morocco'
ORDER BY Year
"""
pd.read_sql(query, conn)

,Entity,Year,Cardiovascular diseases,avg_cardio
0,Morocco,1990,57888.0,85037.0
1,Morocco,1991,59719.0,85037.0
2,Morocco,1992,62875.0,85037.0
3,Morocco,1993,65445.0,85037.0
4,Morocco,1994,66526.0,85037.0
5,Morocco,1995,68162.0,85037.0
6,Morocco,1996,69194.0,85037.0
7,Morocco,1997,70916.0,85037.0
8,Morocco,1998,72395.0,85037.0
9,Morocco,1999,74713.0,85037.0


Rank countries by their total Cardiovascular deaths in 2019 (rank 1 = highest). Show Entity, deaths, and rank. Then filter to show only the top 10 ranked countries.

In [148]:
query = """ 

SELECT 
    Entity, 
    Year, 
    "Cardiovascular diseases",
    RANK() OVER (ORDER BY "Cardiovascular diseases" DESC) AS rank
FROM causes_of_death
WHERE Code NOT LIKE 'OWID%' AND Year = 2019
LIMIT 10

"""
pd.read_sql(query, conn)

,Entity,Year,Cardiovascular diseases,rank
0,China,2019,4584273.0,1
1,India,2019,2574410.0,2
2,Russia,2019,1004931.0,3
3,United States,2019,957455.0,4
4,Indonesia,2019,651481.0,5
5,Ukraine,2019,449376.0,6
6,Brazil,2019,397993.0,7
7,Japan,2019,372483.0,8
8,Germany,2019,364285.0,9
9,Pakistan,2019,341108.0,10


For each country, show how Cardiovascular deaths changed year over year using LAG(). Show Entity, Year, current year deaths, previous year deaths, and the difference. Filter for Morocco.

In [156]:
query = """ 

SELECT
    Entity,
    Year,
    "Cardiovascular diseases",
    LAG("Cardiovascular diseases", 1, 0) OVER (ORDER BY Year) AS previous_year_deaths,
    "Cardiovascular diseases" - LAG("Cardiovascular diseases", 1, 0) OVER (ORDER BY Year) AS difference
FROM causes_of_death
WHERE Entity = 'Morocco'


"""
pd.read_sql(query, conn)

,Entity,Year,Cardiovascular diseases,previous_year_deaths,difference
0,Morocco,1990,57888.0,0.0,57888.0
1,Morocco,1991,59719.0,57888.0,1831.0
2,Morocco,1992,62875.0,59719.0,3156.0
3,Morocco,1993,65445.0,62875.0,2570.0
4,Morocco,1994,66526.0,65445.0,1081.0
5,Morocco,1995,68162.0,66526.0,1636.0
6,Morocco,1996,69194.0,68162.0,1032.0
7,Morocco,1997,70916.0,69194.0,1722.0
8,Morocco,1998,72395.0,70916.0,1479.0
9,Morocco,1999,74713.0,72395.0,2318.0


Full pipeline: (1) Using a CTE, calculate total infectious deaths (Malaria + HIV/AIDS + Tuberculosis) AND total chronic deaths (Cardiovascular + Neoplasms) per country per year. (2) Add a window function to rank countries by infectious deaths within each year. (3) Show only rank 1 countries — the most infectious-disease-burdened country each year. Has the same country dominated every year?

In [164]:
query = """

WITH infectious AS (
    SELECT Entity,Year,
        ROUND("Malaria" + "HIV/AIDS" + "Tuberculosis", 0) AS infectious_total,
        ROUND("Cardiovascular diseases" + "Neoplasms", 0) AS chronic_total
    FROM causes_of_death
    WHERE Code NOT LIKE 'OWID%'
),

ranked AS (
    SELECT Entity,Year, infectious_total, chronic_total,
        RANK() OVER ( PARTITION BY Year ORDER BY infectious_total DESC) AS infectious_rank
    FROM infectious

)
SELECT Entity, Year, infectious_total, chronic_total, infectious_rank 
FROM ranked
WHERE infectious_rank = 1 
ORDER BY Year


"""
pd.read_sql(query, conn)

,Entity,Year,infectious_total,chronic_total,infectious_rank
0,India,1990,777506.0,1578462.0,1
1,India,1991,798496.0,1631056.0,1
2,India,1992,812319.0,1669527.0,1
3,India,1993,773594.0,1695990.0,1
4,India,1994,742421.0,1725332.0,1
5,India,1995,728101.0,1739758.0,1
6,India,1996,731118.0,1817168.0,1
7,India,1997,761892.0,1956864.0,1
8,India,1998,769491.0,2004879.0,1
9,India,1999,762846.0,1991786.0,1


Find all countries where Tuberculosis deaths in 2010 were above the global average Tuberculosis deaths in 2010. Show Entity and deaths.

In [172]:
query = """ 

SELECT Entity , Tuberculosis
FROM causes_of_death 
WHERE Code NOT LIKE 'OWID%' AND Year = 2010 AND Tuberculosis > (

    SELECT AVG(Tuberculosis) 
    FROM causes_of_death
    WHERE Code NOT LIKE 'OWID%' AND Year = 2010

)

ORDER BY Tuberculosis DESC
LIMIT 10

"""

pd.read_sql(query, conn)

,Entity,Tuberculosis
0,India,459913.0
1,Indonesia,101631.0
2,Pakistan,72198.0
3,Democratic Republic of Congo,57750.0
4,China,53312.0
5,Nigeria,49386.0
6,Ethiopia,41580.0
7,Bangladesh,37697.0
8,South Africa,31030.0
9,Philippines,28991.0


For each country in 2019, show their Malaria deaths AND how many times larger it is than the global average. Label this 'times_above_avg'. Only show countries above average.

In [180]:
query = """

SELECT Entity , Malaria , "Malaria" * 1.0  /  ROUND (
(
    SELECT AVG("Malaria")
        FROM causes_of_death
        WHERE Year = 2019 AND Code NOT LIKE 'OWID%'
),0)  AS times_above_avg 

FROM  causes_of_death
WHERE Year = 2019 AND Code NOT LIKE 'OWID%' AND Malaria > (SELECT AVG(Malaria) FROM causes_of_death WHERE Year = 2019 AND Code NOT LIKE 'OWID%')
ORDER BY times_above_avg DESC

"""
pd.read_sql(query, conn)

,Entity,Malaria,times_above_avg
0,Nigeria,191106.0,60.610847
1,Democratic Republic of Congo,57160.0,18.128766
2,India,33372.0,10.584206
3,Cote d'Ivoire,29398.0,9.323819
4,Burkina Faso,26305.0,8.342848
5,Niger,23822.0,7.555344
6,Cameroon,22652.0,7.184269
7,Uganda,22587.0,7.163654
8,Ghana,21597.0,6.849667
9,Mozambique,20528.0,6.510625



Find the country with the LOWEST Cardiovascular deaths in 2019 (excluding zeros). Use a subquery — not ORDER BY + LIMIT.

In [216]:
query = """

SELECT Entity, Year , "Cardiovascular diseases" AS deaths
FROM causes_of_death
WHERE Year = 2019 AND Code NOT LIKE 'OWID%' AND "Cardiovascular diseases" = 
(
      SELECT MIN("Cardiovascular diseases") 
      FROM causes_of_death
      WHERE Year = 2019 AND Code NOT LIKE 'OWID%' AND "Cardiovascular diseases" > 0
)

"""
pd.read_sql(query, conn)

,Entity,Year,deaths
0,Tokelau,2019,4.0


Find years where global Neonatal disorders deaths were BELOW the all-time average across all years. Show Year and total deaths. Is neonatal mortality improving?

In [231]:
query= """ 

SELECT Year , SUM("Neonatal disorders")
FROM causes_of_death
WHERE Code NOT LIKE 'OWID%'
GROUP BY Year
HAVING SUM("Neonatal disorders") < (
    SELECT AVG(total)
    FROM (
        SELECT SUM("Neonatal disorders") AS total
        FROM causes_of_death
        WHERE Code NOT LIKE 'OWID%'
        GROUP BY Year
    )
)

"""
pd.read_sql(query, conn)

,Year,"SUM(""Neonatal disorders"")"
0,2007,2549183.0
1,2008,2515774.0
2,2009,2475066.0
3,2010,2428938.0
4,2011,2383663.0
5,2012,2339980.0
6,2013,2299343.0
7,2014,2237471.0
8,2015,2183144.0
9,2016,2099080.0


For Morocco specifically, find years where Cardiovascular deaths were above Morocco's OWN average. Show Year and deaths. How many years were above average?

In [234]:
query = """ 

SELECT Year, "Cardiovascular diseases"
FROM causes_of_death
WHERE Entity = 'Morocco' 
AND "Cardiovascular diseases" > (
    SELECT AVG("Cardiovascular diseases")
    FROM causes_of_death
    WHERE Entity = 'Morocco'
    )
"""
pd.read_sql(query, conn)

,Year,Cardiovascular diseases
0,2006,85163.0
1,2007,87876.0
2,2008,89746.0
3,2009,91495.0
4,2010,93006.0
5,2011,94564.0
6,2012,96921.0
7,2013,107171.0
8,2014,108079.0
9,2015,109198.0



Find countries where their 2019 HIV/AIDS deaths were MORE THAN DOUBLE their 2000 HIV/AIDS deaths. Show Entity, 2000 deaths, 2019 deaths, and the ratio.

In [241]:
query = """ 

WITH deaths_2000 AS (

SELECT Entity, "HIV/AIDS" AS hiv_2000
FROM causes_of_death
WHERE Year = 2000 AND Code NOT LIKE 'OWID%'

),
deaths_2019 AS (

SELECT Entity, "HIV/AIDS" AS hiv_2019
FROM causes_of_death
WHERE Year = 2019 AND Code NOT LIKE 'OWID%'

)
SELECT c1.Entity , c1.hiv_2000 , c2.hiv_2019 , ROUND(c2.hiv_2019 * 1.0 / NULLIF(c1.hiv_2000, 0), 2) AS ratio
FROM deaths_2000 AS c1
JOIN deaths_2019 AS c2

ON c1.Entity = c2.Entity
WHERE c2.hiv_2019 > c1.hiv_2000 * 2


"""
pd.read_sql(query, conn)

,Entity,hiv_2000,hiv_2019,ratio
0,Afghanistan,97.0,318.0,3.28
1,Algeria,123.0,264.0,2.15
2,Angola,4784.0,16802.0,3.51
3,Armenia,1.0,19.0,19.00
4,Bangladesh,19.0,359.0,18.89
5,Brunei,2.0,5.0,2.50
6,China,9872.0,31746.0,3.22
7,Comoros,0.0,2.0,NaN
8,Cook Islands,0.0,2.0,NaN
9,Cuba,135.0,337.0,2.50


Using a CTE, calculate the total deaths from ALL causes per country per year (sum every death column). Then show the top 5 deadliest countries in 2019.

In [245]:
query = """ 

WITH total_deaths AS(
SELECT Entity, Year,
        ROUND(
            "Meningitis" + "Neoplasms" + "Malaria" + 
            "Drowning" + "HIV/AIDS" + "Tuberculosis" + 
            "Neonatal disorders" + "Diarrheal diseases" + 
            "Cardiovascular diseases" + "Diabetes mellitus" + 
            "Chronic respiratory diseases" + "Self-harm" +
            "Lower respiratory infections" + "Nutritional deficiencies"
        , 0) AS all_cause_total
    FROM causes_of_death
    WHERE Code NOT LIKE 'OWID%'
)
SELECT Entity, Year, all_cause_total
FROM total_deaths
WHERE Year = 2019
ORDER BY all_cause_total DESC
LIMIT 10


"""
pd.read_sql(query, conn)

,Entity,Year,all_cause_total
0,China,2019,9064024.0
1,India,2019,7265767.0
2,United States,2019,2202930.0
3,Russia,2019,1471228.0
4,Indonesia,2019,1350820.0
5,Nigeria,2019,1223733.0
6,Pakistan,2019,1170872.0
7,Japan,2019,1041374.0
8,Brazil,2019,982746.0
9,Germany,2019,750402.0


Using a CTE, find each country's average Malaria deaths across all years. Then in the main query, show only countries whose average is above the MEDIAN of all country averages. (Hint: use the middle value approach)

In [253]:
query = """ 

WITH malaria_deaths AS (
SELECT Entity ,  AVG("Malaria") as avg_malaria
FROM causes_of_death 
WHERE Code NOT LIKE 'OWID%' 
GROUP BY Entity
)
SELECT Entity, ROUND(avg_malaria, 0) AS avg_malaria 
FROM malaria_deaths 
WHERE avg_malaria > (SELECT AVG(avg_malaria) FROM malaria_deaths )
ORDER BY avg_malaria DESC
""" 
pd.read_sql(query, conn)

,Entity,avg_malaria
0,Nigeria,214069.0
1,Democratic Republic of Congo,85241.0
2,India,81308.0
3,Uganda,42188.0
4,Burkina Faso,31692.0
5,Cote d'Ivoire,31387.0
6,Mozambique,27265.0
7,Tanzania,26683.0
8,Ghana,24045.0
9,Mali,23703.0


Using THREE CTEs: (1) total Cardiovascular deaths per country, (2) total Malaria deaths per country, (3) total Diabetes deaths per country. JOIN all three and show countries where cardiovascular is the dominant cause (largest of the three).

In [255]:
query = """

WITH cardio AS (

SELECT Entity , SUM("Cardiovascular diseases") AS total_cardio
FROM causes_of_death 
WHERE Code NOT LIKE 'OWID%' 
GROUP BY Entity

),
malaria AS(

SELECT Entity , SUM("Malaria") AS total_malaria
FROM causes_of_death 
WHERE Code NOT LIKE 'OWID%' 
GROUP BY Entity

),
diabetes AS(

SELECT Entity , SUM("Diabetes mellitus") AS total_diabetes
FROM causes_of_death 
WHERE Code NOT LIKE 'OWID%' 
GROUP BY Entity
)
SELECT c.Entity , c.total_cardio  , m.total_malaria , d.total_diabetes
FROM cardio AS c
JOIN malaria AS m ON c.Entity = m.Entity 
JOIN diabetes AS d ON c.Entity = d.Entity
WHERE c.total_cardio >  m.total_malaria 
    AND c.total_cardio > d.total_diabetes

ORDER BY c.total_cardio DESC
LIMIT 15
"""
pd.read_sql(query, conn)

,Entity,total_cardio,total_malaria,total_diabetes
0,China,100505973.0,13419.0,3468554.0
1,India,52994710.0,2439244.0,4781170.0
2,Russia,33903781.0,0.0,369649.0
3,United States,26438346.0,0.0,2030632.0
4,Indonesia,13587012.0,74664.0,1888549.0
5,Ukraine,13053052.0,0.0,95725.0
6,Germany,10819770.0,0.0,621574.0
7,Brazil,9589019.0,39970.0,1315755.0
8,Japan,9210437.0,0.0,251165.0
9,Pakistan,7745192.0,213590.0,845343.0


Using CTEs, find Morocco's year with the highest Cardiovascular deaths AND year with the lowest. Show both in one result with a label ('Peak Year' and 'Best Year').

In [267]:
query = """

WITH country AS (
SELECT Year, ROUND("Cardiovascular diseases" ,0) AS cardio
FROM causes_of_death 
WHERE Entity = 'Morocco'
),
Peak AS (
SELECT Year, cardio, 'Peak Year' AS label
FROM country
WHERE cardio = (SELECT MAX(cardio) FROM country)
),
Best AS (
SELECT Year , cardio , 'Best Year' AS label
FROM country
WHERE cardio = (SELECT MIN(cardio) FROM country)
)

SELECT * FROM Peak
UNION ALL
SELECT * FROM Best


"""
pd.read_sql(query, conn)

,Year,cardio,label
0,2019,117034.0,Peak Year
1,1990,57888.0,Best Year


Using a CTE, calculate the percentage each cause contributes to total deaths for each country in 2019. Focus on Cardiovascular diseases, Malaria, and HIV/AIDS. Show Entity and percentages.

In [276]:
query = """ 

WITH cte AS(

SELECT Entity, "Cardiovascular diseases" + "Malaria" + "HIV/AIDS" + "Tuberculosis" + "Neonatal disorders" + "Diabetes mellitus" AS total
FROM causes_of_death 
WHERE Year = 2019 AND Code NOT LIKE 'OWID%' 
)
SELECT d.Entity , ROUND("Cardiovascular diseases" * 100 /c.total,1) AS p_cardio ,
                ROUND("Malaria" * 100 /c.total,1) AS p_malaria , 
                ROUND("HIV/AIDS" * 100 /c.total,1) AS p_hiv 
                
FROM causes_of_death AS d 
JOIN cte AS c 
ON c.Entity = d.Entity
WHERE d.Year = 2019 AND d.Code NOT LIKE 'OWID%'
LIMIT 15


"""
pd.read_sql(query, conn)

,Entity,p_cardio,p_malaria,p_hiv
0,Afghanistan,65.3,0.6,0.3
1,Albania,97.4,0.0,0.0
2,Algeria,86.9,0.0,0.2
3,American Samoa,71.8,0.0,0.0
4,Andorra,93.4,0.0,1.7
5,Angola,29.5,12.4,19.2
6,Antigua and Barbuda,74.6,0.0,2.6
7,Argentina,86.0,0.0,1.5
8,Armenia,90.1,0.0,0.1
9,Australia,91.3,0.0,0.1


Using CTEs, find countries that went from HIGH Malaria burden (above global average) in 2000 to LOW Malaria burden (below global average) in 2019. These are countries that successfully reduced Malaria.

In [298]:
query = """

WITH high_burden AS (

SELECT Entity , "Malaria" AS malaria_2000
FROM causes_of_death
WHERE Year = 2000 AND Code NOT LIKE 'OWID%' 
AND "Malaria" > ( SELECT AVG("Malaria") FROM causes_of_death WHERE Year = 2000 AND Code NOT LIKE 'OWID%')
),
low_burden AS (

SELECT Entity , "Malaria" AS malaria_2019
FROM causes_of_death
WHERE Year = 2019 AND Code NOT LIKE 'OWID%' 
AND "Malaria" < ( SELECT AVG("Malaria") FROM causes_of_death WHERE Year = 2019 AND Code NOT LIKE 'OWID%')
)

SELECT h.Entity , h.malaria_2000 , l.malaria_2019 , ROUND((h.malaria_2000 - l.malaria_2019) * 100.0 / h.malaria_2000, 1) AS Per_cent
FROM  high_burden AS h
JOIN  low_burden AS l
ON h.Entity = l.Entity
ORDER BY Per_cent DESC


"""
pd.read_sql(query, conn)

,Entity,malaria_2000,malaria_2019,Per_cent
0,Bangladesh,8967.0,110.0,98.8
1,Zimbabwe,5353.0,2068.0,61.4
2,Sudan,5024.0,2337.0,53.5


For each country, show each year's Cardiovascular deaths AND a running total (cumulative sum) of deaths up to that year. Limit to Morocco.

In [302]:
query = """ 

SELECT Entity, Year, "Cardiovascular diseases", SUM("Cardiovascular diseases") OVER(PARTITION BY Entity ORDER BY Year) AS cumulativa_sum
FROM causes_of_death
WHERE Entity = 'Morocco'
ORDER BY Year


"""
pd.read_sql(query, conn)

,Entity,Year,Cardiovascular diseases,cumulativa_sum
0,Morocco,1990,57888.0,57888.0
1,Morocco,1991,59719.0,117607.0
2,Morocco,1992,62875.0,180482.0
3,Morocco,1993,65445.0,245927.0
4,Morocco,1994,66526.0,312453.0
5,Morocco,1995,68162.0,380615.0
6,Morocco,1996,69194.0,449809.0
7,Morocco,1997,70916.0,520725.0
8,Morocco,1998,72395.0,593120.0
9,Morocco,1999,74713.0,667833.0


For each year, rank ALL countries by Malaria deaths. Then show only countries ranked 1st, 2nd, and 3rd for year 2010. Use DENSE_RANK instead of RANK.

In [307]:
query = """

WITH ranked AS (

SELECT Entity, Year, "Malaria" , DENSE_RANK() OVER (PARTITION BY Year ORDER BY "Malaria" DESC) AS rank
FROM causes_of_death
WHERE Code NOT LIKE 'OWID%' 


)
SELECT Entity, Year, "Malaria" ,rank 
FROM ranked 
WHERE Year = 2010 AND rank <= 3
ORDER BY rank



"""
pd.read_sql(query, conn)

,Entity,Year,Malaria,rank
0,Nigeria,2010,266638.0,1
1,Democratic Republic of Congo,2010,91494.0,2
2,India,2010,60236.0,3



For Morocco, show each year's Cardiovascular deaths alongside the NEXT year's deaths using LEAD(). Also calculate the difference. Which year had the biggest increase coming up?

In [314]:
query= """ 

SELECT
    Entity,
    Year,
    "Cardiovascular diseases",
    LEAD("Cardiovascular diseases", 1, 0) OVER (PARTITION BY Entity ORDER BY Year) AS next_year_deaths,
    LEAD("Cardiovascular diseases", 1, 0) OVER (PARTITION BY Entity ORDER BY Year) - "Cardiovascular diseases"  AS difference
FROM causes_of_death
WHERE Entity = 'Morocco'


"""

pd.read_sql(query, conn)

,Entity,Year,Cardiovascular diseases,next_year_deaths,difference
0,Morocco,1990,57888.0,59719.0,1831.0
1,Morocco,1991,59719.0,62875.0,3156.0
2,Morocco,1992,62875.0,65445.0,2570.0
3,Morocco,1993,65445.0,66526.0,1081.0
4,Morocco,1994,66526.0,68162.0,1636.0
5,Morocco,1995,68162.0,69194.0,1032.0
6,Morocco,1996,69194.0,70916.0,1722.0
7,Morocco,1997,70916.0,72395.0,1479.0
8,Morocco,1998,72395.0,74713.0,2318.0
9,Morocco,1999,74713.0,73717.0,-996.0


For each country, calculate a 3-year moving average of Cardiovascular deaths. Show Entity, Year, actual deaths, and the moving average. Limit to Morocco.

In [321]:
query = """

SELECT Entity, Year, "Cardiovascular diseases" AS actual_deaths,
    ROUND(AVG("Cardiovascular diseases") OVER ( PARTITION BY Entity ORDER BY Year ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 0) AS moving_avg_3yr
FROM causes_of_death
WHERE Entity = 'Morocco'
ORDER BY Year
"""
pd.read_sql(query, conn)

,Entity,Year,actual_deaths,moving_avg_3yr
0,Morocco,1990,57888.0,57888.0
1,Morocco,1991,59719.0,58804.0
2,Morocco,1992,62875.0,60161.0
3,Morocco,1993,65445.0,62680.0
4,Morocco,1994,66526.0,64949.0
5,Morocco,1995,68162.0,66711.0
6,Morocco,1996,69194.0,67961.0
7,Morocco,1997,70916.0,69424.0
8,Morocco,1998,72395.0,70835.0
9,Morocco,1999,74713.0,72675.0


Using ROW_NUMBER(), assign a unique number to each country within each year based on Tuberculosis deaths (highest = 1). Then show only row_number = 1 for each year from 2010 to 2019. Has the same country been #1 every year?

In [323]:
query = """
WITH numbered AS (
    SELECT Entity, Year, "Tuberculosis" AS tb_deaths,
           ROW_NUMBER() OVER ( PARTITION BY Year ORDER BY "Tuberculosis" DESC
           ) AS rank
    FROM causes_of_death
    WHERE Code NOT LIKE 'OWID%'
)
SELECT Entity, Year, tb_deaths, rank
FROM numbered
WHERE rank = 1 AND Year BETWEEN 2010 AND 2019
ORDER BY Year
"""
pd.read_sql(query, conn)

,Entity,Year,tb_deaths,rank
0,India,2010,459913.0,1
1,India,2011,453408.0,1
2,India,2012,461592.0,1
3,India,2013,466722.0,1
4,India,2014,462649.0,1
5,India,2015,445845.0,1
6,India,2016,440225.0,1
7,India,2017,440926.0,1
8,India,2018,434157.0,1
9,India,2019,422634.0,1


For each country, show the year-over-year PERCENTAGE CHANGE in Cardiovascular deaths using LAG(). Show Entity, Year, current deaths, previous deaths, and pct_change. Filter for Morocco and only show years where pct_change was negative (deaths decreased).

In [324]:
query = """
WITH yoy AS (
    SELECT Entity, Year, "Cardiovascular diseases" AS current_deaths,
           ROUND(LAG("Cardiovascular diseases") OVER (PARTITION BY Entity ORDER BY Year), 0) AS prev_deaths,
           ROUND(("Cardiovascular diseases" - LAG("Cardiovascular diseases") OVER ( PARTITION BY Entity ORDER BY Year)) 
               * 100.0 / NULLIF(LAG("Cardiovascular diseases") OVER (PARTITION BY Entity ORDER BY Year), 0), 2) AS pct_change
    FROM causes_of_death
    WHERE Entity = 'Morocco'
)
SELECT * FROM yoy
WHERE pct_change < 0
ORDER BY pct_change ASC
"""
pd.read_sql(query, conn)

,Entity,Year,current_deaths,prev_deaths,pct_change
0,Morocco,2000,73717.0,74713.0,-1.33


For year 2019, show each country's Cardiovascular deaths AND what PERCENTILE they fall in (0-100). Use PERCENT_RANK() window function. Which percentile is Morocco in?

In [325]:
query = """
WITH percentiles AS (
    SELECT Entity, "Cardiovascular diseases" AS cardio_deaths, ROUND(PERCENT_RANK() OVER ( ORDER BY "Cardiovascular diseases" ) * 100, 1) AS percentile
    FROM causes_of_death
    WHERE Year = 2019 AND Code NOT LIKE 'OWID%'
)
SELECT *
FROM percentiles
WHERE Entity = 'Morocco' OR percentile >= 99
ORDER BY percentile DESC
"""
pd.read_sql(query, conn)

,Entity,cardio_deaths,percentile
0,China,4584273.0,100.0
1,India,2574410.0,99.5
2,Russia,1004931.0,99.0
3,Morocco,117034.0,87.7


Find countries that were consistently in the TOP 10 for Cardiovascular deaths for EVERY year from 2010 to 2019. A country must appear in the top 10 ALL 10 years.

In [326]:
query = """
WITH ranked AS (
    SELECT Entity, Year,
           RANK() OVER (
               PARTITION BY Year 
               ORDER BY "Cardiovascular diseases" DESC
           ) AS cardio_rank
    FROM causes_of_death
    WHERE Code NOT LIKE 'OWID%'
      AND Year BETWEEN 2010 AND 2019
),
top10_per_year AS (
    SELECT Entity, Year
    FROM ranked
    WHERE cardio_rank <= 10
)
SELECT Entity, COUNT(*) AS years_in_top10
FROM top10_per_year
GROUP BY Entity
HAVING COUNT(*) = 10
ORDER BY Entity
"""
pd.read_sql(query, conn)

,Entity,years_in_top10
0,Brazil,10
1,China,10
2,Germany,10
3,India,10
4,Indonesia,10
5,Japan,10
6,Pakistan,10
7,Russia,10
8,Ukraine,10
9,United States,10


For each country, find the year where Malaria deaths dropped the MOST compared to the previous year (biggest single-year reduction). Show Entity, Year, the drop amount, and rank countries by biggest drop.

In [327]:
query = """
WITH yoy AS (
    SELECT Entity, Year,
           "Malaria" AS malaria,
           LAG("Malaria") OVER (
               PARTITION BY Entity ORDER BY Year
           ) AS prev_malaria,
           "Malaria" - LAG("Malaria") OVER (
               PARTITION BY Entity ORDER BY Year
           ) AS yoy_change
    FROM causes_of_death
    WHERE Code NOT LIKE 'OWID%'
),
biggest_drop AS (
    SELECT Entity, Year, 
           ROUND(malaria, 0) AS malaria_deaths,
           ROUND(yoy_change, 0) AS annual_change,
           ROW_NUMBER() OVER (
               PARTITION BY Entity 
               ORDER BY yoy_change ASC
           ) AS drop_rank
    FROM yoy
    WHERE yoy_change IS NOT NULL
)
SELECT Entity, Year, malaria_deaths, annual_change
FROM biggest_drop
WHERE drop_rank = 1
  AND annual_change < 0
ORDER BY annual_change ASC
LIMIT 15
"""
pd.read_sql(query, conn)

,Entity,Year,malaria_deaths,annual_change
0,India,1999,84608.0,-23539.0
1,Nigeria,2016,186482.0,-18190.0
2,Senegal,2001,7662.0,-12638.0
3,Bangladesh,2000,8967.0,-9872.0
4,Ethiopia,2005,29768.0,-9268.0
5,Democratic Republic of Congo,2012,78228.0,-7368.0
6,Uganda,2012,27954.0,-7145.0
7,Mali,2016,21363.0,-6527.0
8,Pakistan,2013,4013.0,-5834.0
9,Tanzania,2004,29536.0,-4940.0



FINAL BOSS: For each country, calculate: (1) total infectious deaths, (2) total chronic deaths, (3) rank by infectious burden, (4) rank by chronic burden, (5) identify countries where infectious rank is much higher than chronic rank (infectious disease dominates relative to chronic). Show top 15 most "infectious-dominant" countries.

In [328]:
query = """
WITH disease_burden AS (
    SELECT Entity,
           ROUND(SUM("Malaria" + "HIV/AIDS" + "Tuberculosis" + 
                     "Neonatal disorders"), 0) AS infectious_total,
           ROUND(SUM("Cardiovascular diseases" + "Neoplasms" + 
                     "Diabetes mellitus"), 0) AS chronic_total
    FROM causes_of_death
    WHERE Code NOT LIKE 'OWID%'
    GROUP BY Entity
),
ranked AS (
    SELECT Entity, infectious_total, chronic_total,
           RANK() OVER (ORDER BY infectious_total DESC) AS infectious_rank,
           RANK() OVER (ORDER BY chronic_total DESC) AS chronic_rank
    FROM disease_burden
)
SELECT 
    Entity,
    infectious_total,
    chronic_total,
    infectious_rank,
    chronic_rank,
    chronic_rank - infectious_rank AS rank_gap
FROM ranked
WHERE infectious_total > 0 AND chronic_total > 0
ORDER BY rank_gap DESC
LIMIT 15
"""
pd.read_sql(query, conn)

,Entity,infectious_total,chronic_total,infectious_rank,chronic_rank,rank_gap
0,Burundi,1137117.0,415569.0,29,118,89
1,Central African Republic,795056.0,254116.0,45,134,89
2,Lesotho,445927.0,182519.0,55,142,87
3,Niger,1461693.0,492746.0,26,112,86
4,Sierra Leone,764700.0,302109.0,46,131,85
5,Togo,557959.0,251951.0,51,135,84
6,Botswana,321450.0,144440.0,65,149,84
7,South Sudan,714094.0,314205.0,47,129,82
8,Liberia,327642.0,163586.0,62,144,82
9,Eswatini,195809.0,75826.0,75,157,82
